In [5]:
MODEL_ID = 'moonshotai/kimi-k2-instruct-0905'
EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"
import os
from dotenv import load_dotenv
load_dotenv("../keys.env")
assert os.environ["GROQ_API_KEY"].startswith("gsk"), \
  "Please specify the GROQ_API_KEY access token in keys.env file"
assert os.environ["HF_TOKEN"].startswith("hf"), \
  "Please specifuy HF_TOKEN environment variable in keys.env file"  

In [6]:
import sys
sys.path.append('../06_basic_rag')
import gutenberg_text_loader as gtl
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s -%(message)s')
logger = logging.getLogger(__name__)

In [7]:
!ls ./.cache vector_index

./.cache:
pg46976_1b16d8525c.txt

vector_index:
default__vector_store.json  graph_store.json	      index_store.json
docstore.json		    image__vector_store.json


In [8]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core import Document

INDEX_DIR = 'vector_index'
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_ID)

Settings.chunk_size = 1024
Settings.chunk_overlap = 20
TOP_K = 2

if os.path.isdir(INDEX_DIR):
    print("Loading in already created index")
    storage_context = StorageContext.from_defaults(persist_dir=INDEX_DIR)
    index = load_index_from_storage(storage_context)
else:
    gs = gtl.GutenbergSource()
    doc = gs.load_from_url("https://www.gutenberg.org/cache/epub/46976/pg46976.txt")
    # Read all files in ./cache
    documents = SimpleDirectoryReader(input_dir='./.cache', required_exts=['.txt'], exclude_hidden=False).load_data()
    # creates a vector db
    index = VectorStoreIndex.from_documents(documents)
    index.storage_context.persist(persist_dir=INDEX_DIR)


2026-04-09 10:51:01,131 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
2026-04-09 10:51:06,281 - INFO - 1 prompt is loaded, with the key: query


Loading in already created index


2026-04-09 10:51:07,416 - INFO - Loading all indices.


In [9]:
from llama_index.llms.groq import Groq
from llama_index.core.query_engine import RetrieverQueryEngine

llm = Groq(model=MODEL_ID, api_key=os.environ['GROQ_API_KEY'])

query_engine = RetrieverQueryEngine.from_args(
    retriever=index.as_retriever(similarity_top_k=TOP_K), llm=llm,
)

def semantic_rag(question):
    response = query_engine.query(question)
    response = {"answer": str(response),
                "source_nodes": response.source_nodes
                }
    print(response['answer'])

    for node in response['source_nodes']:
        print(node)
    return response

semantic_rag("How did Alexander treat the people of the places he conquered?");

2026-04-09 10:51:10,327 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Alexander’s treatment varied by circumstance. He could be ruthless: at Thebes he razed the city, enslaved survivors, and confiscated land. Yet he also showed restraint and favor: he spared Pindar’s house and descendants, pardoned most Athenian opponents, and at Ephesus prevented massacres while restoring democratic government and remitting tribute.
Node ID: b61866c8-3547-4999-afe1-a8bb32a57373
Text: They resolved to occupy the  Cadmea with a garrison; to raze the
city to the ground; to distribute  among themselves all the territory,
except what was dedicated to the  gods; and to sell into slavery the
women and children, and as many of  the males as survived, except
those who were priests or priestesses,  and those who were bound to
Philip o...
Score:  0.741

Node ID: 7593b199-03d5-403d-884b-6f13b05ce447
Text: When the people of  Ephesus were relieved of their dread of the
oligarchs, they rushed  headlong to kill the men who had brought
Memnon into the city, as also  those who had pilla

In [10]:
semantic_rag("Where did Alexander die?")

2026-04-09 10:51:11,561 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


He died in the palace at Babylon, remaining where he was rather than being carried into the temple of Serapis.
Node ID: f1c31658-1815-4de4-83db-53b3044e6f77
Text: ALEXANDER’S DEATH.      Such is the account given in the Royal
Diary. In addition to this, it  states that the soldiers were very
desirous of seeing him; some, in  order to see him once more while
still alive; others, because there was  a report that he was already
dead, imagined that his death was being  concealed by the confidential
body-guard...
Score:  0.720

Node ID: 0a9129e3-e820-42fb-ad1d-22ac58d0bcfd
Text: 26, 16); διαπεσούσης αὐτῷ τῆς  ἐπιβουλῆς.    [565] Alexander
wrote to Craterus, Attalus, and Alcetas, that the  pages, though put
to the torture, asserted that no one but themselves  was privy to the
conspiracy. In another letter, written to Antipater  the regent of
Macedonia, he says that the pages had been stoned  to death by the
Macedonians, ...
Score:  0.703



{'answer': 'He died in the palace at Babylon, remaining where he was rather than being carried into the temple of Serapis.',
 'source_nodes': [NodeWithScore(node=TextNode(id_='f1c31658-1815-4de4-83db-53b3044e6f77', embedding=None, metadata={'file_path': '/home/sharofiddin/practice/py3/gen_ai_design_patterns/09_index_aware_retreival/.cache/pg46976_1b16d8525c.txt', 'file_name': 'pg46976_1b16d8525c.txt', 'file_type': 'text/plain', 'file_size': 920246, 'creation_date': '2026-04-08', 'last_modified_date': '2026-04-08'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='35564ce1-e827-4026-a377-4abf43ccb1f8', node_type='4', metadata={'file_path': '/home/sharofiddin/practice/py3/gen_ai_design_patterns/09_index_aware_

In [11]:
semantic_rag("Describe the relationship between Alexander and Diogenes.");


2026-04-09 10:54:36,411 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Alexander approached Diogenes with the deference due a renowned philosopher, offering to grant any request. Diogenes replied only that Alexander and his entourage should move aside and stop blocking the sunlight. Instead of taking offense, Alexander openly admired Diogenes’s independence and composure, indicating a respectful, if distant, rapport in which the conqueror acknowledged the philosopher’s moral authority.
Node ID: 106db555-7a08-44ba-b8a5-e83008faa9f5
Text: [831] And yet thou also wilt soon die, and possess only as much
of the earth as is sufficient for thy body to be buried in.”
CHAPTER II.    ALEXANDER’S DEALINGS WITH THE INDIAN SAGES.      On
this occasion Alexander commended both the words and the men who
spoke them; but nevertheless he did just the opposite to that which he
commend...
Score:  0.733

Node ID: 775a9309-b0b4-4ce1-809e-074a8249c5f5
Text: [831] Cf. Alciphron (_Epistolae_, i. 30, 1), with Bergler and
Wagner’s  notes.    [832] This must have occurred B.C. 336. 

In [18]:
semantic_rag("What was Alexander's strategy against Darius III?");

2026-04-09 11:25:21,252 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Alexander’s approach was to seize the tactical high ground first, then force a river crossing under covering fire from siege engines and archers, thereby turning the enemy’s flank and launching a surprise assault on their unfortified camp.
Node ID: 7d8e28f8-95e9-43f9-8103-d6e721dc9f82
Text: The enemy being emboldened, as if the Macedonians  had already
given way, rushed upon them with a run and with no kind  of order. But
when the arrows began to reach them, Alexander at once  wheeled round
at the appointed signal, and led his phalanx against  them with a run.
His horse-lancers, Agrianians, and archers first ran  forward and
engage...
Score:  0.756

Node ID: eb13d844-6c20-4673-bee2-6bfb9d6b7f96
Text: As Alexander saw only a few of the enemy still occupying a
ridge,  along which lay his route, he ordered his body-guards and
personal  companions to take their shields, mount their horses, and
ride to  the hill; and when they reached it, if those who had occupied
the  position awaited them

In [13]:
from llama_index.core.llms import ChatMessage
def create_hypothetical_answer(question):
    messages = [
        ChatMessage(role="system",
                    content="""Answer the following question in 2-3 sentences.
                    If you don't know the answer, make an educated guess.
                    """),
        ChatMessage(role="user", content=question)
    ]
    answer = str(llm.chat(messages))
    return answer

def hyde_rag(question):
    answer = create_hypothetical_answer(question)
    print("Hypothetical answer: ", answer)
    return semantic_rag(answer)


In [17]:
hyde_rag("What was Alexander's strategy against Darius III?");

2026-04-09 11:24:41,454 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Hypothetical answer:  assistant: Alexander avoided a head-on clash with Darius’s huge infantry; instead he thinned his center to bait the Persians, then drove a cavalry wedge straight at Darius himself at Issus and Gaugamela, forcing the king to flee and collapsing Persian morale.


2026-04-09 11:24:42,877 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Alexander exploited a gap that opened when Persian cavalry rode around his right; he wheeled the Companions and part of the phalanx into a wedge and charged straight at Darius, whose own flight broke Persian morale and turned the battle.
Node ID: b9ef262a-d3e2-43c6-8447-bdc04b5869d1
Text: But these also were afterwards overpowered by the grooms of
Alexander’s army and by the royal shield-bearing guards.[414]
CHAPTER XIV.    BATTLE OF ARBELA.—FLIGHT OF DARIUS.      As soon as
Darius began to set his whole phalanx in motion, Alexander  ordered
Aretes to attack those who were riding completely round his  right
wing; and up...
Score:  0.801

Node ID: 770d0258-d530-4b0f-bb1f-de42be85c43c
Text: Alexander, however,  led his forces towards the city; and the
enemy, after sacrificing three  boys, an equal number of girls, and
three black rams, sallied forth for  the purpose of receiving the
Macedonians in a hand-to-hand conflict.  But as soon as they came to
close quarters, they left the positio

In [19]:
hyde_rag("Describe the relationship between Alexander and Diogenes.");

2026-04-09 11:27:16,758 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Hypothetical answer:  assistant: Alexander the Great admired Diogenes’ fearless candor; when he offered to grant the Cynic any wish, Diogenes merely asked him to stop blocking the sunlight, prompting Alexander to remark that if he were not Alexander, he would choose to be Diogenes.


2026-04-09 11:27:18,033 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Alexander the Great admired Diogenes’ fearless candor; when he offered to grant the Cynic any wish, Diogenes merely asked him to stop blocking the sunlight, prompting Alexander to remark that if he were not Alexander, he would choose to be Diogenes.
Node ID: 106db555-7a08-44ba-b8a5-e83008faa9f5
Text: [831] And yet thou also wilt soon die, and possess only as much
of the earth as is sufficient for thy body to be buried in.”
CHAPTER II.    ALEXANDER’S DEALINGS WITH THE INDIAN SAGES.      On
this occasion Alexander commended both the words and the men who
spoke them; but nevertheless he did just the opposite to that which he
commend...
Score:  0.778

Node ID: 775a9309-b0b4-4ce1-809e-074a8249c5f5
Text: [831] Cf. Alciphron (_Epistolae_, i. 30, 1), with Bergler and
Wagner’s  notes.    [832] This must have occurred B.C. 336. See
Plutarch (_Alex._ 14);  Cicero (_Tusculanae Disputationes_, v. 32).
Alexander said: “If I were  not Alexander, I should like to be
Diogenes.” Cf. _Arrian_, i. 1;  Plu

In [20]:
semantic_rag("How did the Persian king fight the Greeks?");

2026-04-09 11:29:31,139 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The Persian king relied on Greek mercenaries as his mainstay, arraying them opposite the Macedonian phalanx while his own Persian and Median horse tried to hold the wings; once the cavalry broke, the mercenary infantry stood astonished rather than resolute and were surrounded and destroyed by Alexander’s combined phalanx and cavalry assault.
Node ID: 21d5b2e8-18e0-4ee1-b82a-233474b041f9
Text: He said, moreover, that  the Greeks who were in the two armies
would not be fighting for the  same objects; for those with Darius
were braving danger for pay,  and that pay not high; whereas, those on
their side were voluntarily  defending the interests of Greece. Again,
of foreigners, the Thracians,  Paeonians, Illyrians, and Agrianians,
who we...
Score:  0.725

Node ID: a7f15c46-0473-42b5-85bf-84827cc13984
Text: Showing this to Alexander, he bade him ask some one else for
one. Then  Demaratus, a man of Corinth, one of his personal
Companions, gave him  his own spear; which he had no sooner taken

In [22]:
def add_context_to_query(question):
    messages = [
        ChatMessage(role="system",
                    content="""The following question is about topics discussed in a 2 nd century book about Alexander the Great
                    Clarify the question posed in the following ways:
                    * Expand to include 2 nd century names, For example, a question about Iranians should include answers about Parthians, Persians, Medes, Bactrians, etc.
                    * Provide context on terms. For example, that Ammonites came from Jordan or that Philip was the father of Alexander.
                    Provide only the clarified question without any preamble or instructions.
                    """
                    ),
                    ChatMessage(role="user", content=question)
    ]
    expanded_question = str(llm.chat(messages))
    return expanded_question
def query_expansion_rag(question):
    expanded_question = add_context_to_query(question)
    print("Expanded question: ", expanded_question)
    return semantic_rag(expanded_question)

query_expansion_rag("How did the Persian king fight Greeks?");

2026-04-09 12:00:14,535 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Expanded question:  assistant: How did the Achaemenid Great King—whom 2nd-century writers like Arrian and Curtius call “the Persian,” “the Mede,” or simply “Darius”—fight against the Hellenes, Macedonians, and other Greek-speakers who followed Alexander son of Philip?


2026-04-09 12:00:16,199 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


He confronted the invaders with a composite army whose right wing was led by Mazaeus.  In the decisive set-piece battle Mazaeus launched a vigorous cavalry charge, yet was steadily driven back by Parmenio before Alexander himself arrived on the field.  The Persian centre and left were eventually broken; the king’s forces suffered heavy slaughter—variously reported as 40,000 or more than 90,000 dead—while the Macedonian dead were counted only in the low hundreds.  After the collapse of his line the Great King abandoned the battlefield, leaving the treasures of Babylon to be seized and later distributed by Alexander to his Greek and Macedonian troops.
Node ID: 60aa63ab-2203-490f-96de-0c82451cdf47
Text: He then placed upon the throne one of  his adherents, named
Darius Codomannus, a descendant of one of the  brothers of Artaxerxes
Mnemon. Bagoas soon afterwards tried to poison  this Darius; but the
latter, discovering his treachery, forced him to  drink the deadly
draught himself (_Diod._

In [25]:
from llama_index.core.query_engine import RetrieverQueryEngine
import numpy as np

for alpha in np.arange(0, 1.1, 0.5):
   print(f"***alpha={alpha}")
   query_engine = RetrieverQueryEngine.from_args(
      retriever=index.as_retriever(similarity_top_k=TOP_K),
      llm=llm,
      vector_store_query_mode="hybrid",
      alpha=alpha,
      disable_cache=True
   )

   semantic_rag("Describe the meeting of Alexander and Diogenes")

***alpha=0.0


2026-04-09 12:10:43,914 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Alexander approached Diogenes, who was sunbathing, and asked if he needed anything. Diogenes replied that he only wanted Alexander and his guards to move out of the sunlight. Alexander admired Diogenes’s composure and independence.
Node ID: 775a9309-b0b4-4ce1-809e-074a8249c5f5
Text: [831] Cf. Alciphron (_Epistolae_, i. 30, 1), with Bergler and
Wagner’s  notes.    [832] This must have occurred B.C. 336. See
Plutarch (_Alex._ 14);  Cicero (_Tusculanae Disputationes_, v. 32).
Alexander said: “If I were  not Alexander, I should like to be
Diogenes.” Cf. _Arrian_, i. 1;  Plutarch (_de Fortit. Alex._, p. 331).
[833] Cf. _Strab...
Score:  0.751

Node ID: 106db555-7a08-44ba-b8a5-e83008faa9f5
Text: [831] And yet thou also wilt soon die, and possess only as much
of the earth as is sufficient for thy body to be buried in.”
CHAPTER II.    ALEXANDER’S DEALINGS WITH THE INDIAN SAGES.      On
this occasion Alexander commended both the words and the men who
spoke them; but nevertheless he did just the

2026-04-09 12:10:44,715 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Alexander approached Diogenes, who was sunbathing, and asked if he needed anything. Diogenes replied that he only wanted Alexander and his guards to move out of the sunlight. Alexander admired Diogenes’s response.
Node ID: 775a9309-b0b4-4ce1-809e-074a8249c5f5
Text: [831] Cf. Alciphron (_Epistolae_, i. 30, 1), with Bergler and
Wagner’s  notes.    [832] This must have occurred B.C. 336. See
Plutarch (_Alex._ 14);  Cicero (_Tusculanae Disputationes_, v. 32).
Alexander said: “If I were  not Alexander, I should like to be
Diogenes.” Cf. _Arrian_, i. 1;  Plutarch (_de Fortit. Alex._, p. 331).
[833] Cf. _Strab...
Score:  0.751

Node ID: 106db555-7a08-44ba-b8a5-e83008faa9f5
Text: [831] And yet thou also wilt soon die, and possess only as much
of the earth as is sufficient for thy body to be buried in.”
CHAPTER II.    ALEXANDER’S DEALINGS WITH THE INDIAN SAGES.      On
this occasion Alexander commended both the words and the men who
spoke them; but nevertheless he did just the opposite to that 

2026-04-09 12:10:45,853 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Alexander approached Diogenes, who was lying in the sun, and asked if he needed anything. Diogenes replied that he only wanted Alexander and his attendants to move out of the sunlight. Alexander admired Diogenes’s response.
Node ID: 775a9309-b0b4-4ce1-809e-074a8249c5f5
Text: [831] Cf. Alciphron (_Epistolae_, i. 30, 1), with Bergler and
Wagner’s  notes.    [832] This must have occurred B.C. 336. See
Plutarch (_Alex._ 14);  Cicero (_Tusculanae Disputationes_, v. 32).
Alexander said: “If I were  not Alexander, I should like to be
Diogenes.” Cf. _Arrian_, i. 1;  Plutarch (_de Fortit. Alex._, p. 331).
[833] Cf. _Strab...
Score:  0.751

Node ID: 106db555-7a08-44ba-b8a5-e83008faa9f5
Text: [831] And yet thou also wilt soon die, and possess only as much
of the earth as is sufficient for thy body to be buried in.”
CHAPTER II.    ALEXANDER’S DEALINGS WITH THE INDIAN SAGES.      On
this occasion Alexander commended both the words and the men who
spoke them; but nevertheless he did just the opposit